In [1]:
# ============================================================
# 04_AURORA_ordinal_imbalance_uncertainty_models.ipynb
# AURORA-TWETF Ordinal, Imbalance-Aware, Uncertainty Models
#
# Purpose:
# 1. Load leakage-controlled modeling dataset from Notebook 02.
# 2. Reproduce chronological split used in Notebook 03.
# 3. Exclude invalid previous-label persistence from official comparison.
# 4. Build deployable ordinal and imbalance-aware regime models.
# 5. Add probability calibration and uncertainty diagnostics.
# 6. Save calibrated regime probabilities for later ETF allocation.
#
# Important:
# - This notebook does NOT use y[t-1] future-return labels as features.
# - Valid "persistence-like" information is based only on trailing realized returns.
# - Models are evaluated on 20d and 60d TAIEX ordinal regime targets.
#
# Next notebook:
# 05_AURORA_uncertainty_aware_etf_allocation.ipynb
# ============================================================

from __future__ import annotations

import os
import sys
import json
import time
import math
import random
import hashlib
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        return __import__(import_name)
    except Exception:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return __import__(import_name)

pd = install_if_missing("pandas", "pandas")
np = install_if_missing("numpy", "numpy")
sklearn = install_if_missing("scikit-learn", "sklearn")
matplotlib = install_if_missing("matplotlib", "matplotlib")
seaborn = install_if_missing("seaborn", "seaborn")
joblib = install_if_missing("joblib", "joblib")
pyarrow = install_if_missing("pyarrow", "pyarrow")

try:
    lightgbm = install_if_missing("lightgbm", "lightgbm")
    HAS_LIGHTGBM = True
except Exception as e:
    print("LightGBM unavailable. LightGBM ordinal models will be skipped.")
    print(e)
    HAS_LIGHTGBM = False

try:
    xgboost = install_if_missing("xgboost", "xgboost")
    HAS_XGBOOST = True
except Exception as e:
    print("XGBoost unavailable. XGBoost ordinal models will be skipped.")
    print(e)
    HAS_XGBOOST = False

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestRegressor,
    ExtraTreesRegressor,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    log_loss,
    brier_score_loss,
)
from sklearn.isotonic import IsotonicRegression

if HAS_LIGHTGBM:
    from lightgbm import LGBMClassifier, LGBMRegressor

if HAS_XGBOOST:
    from xgboost import XGBClassifier, XGBRegressor

# ============================================================
# 1. Reproducibility and paths
# ============================================================

RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

PROJECT_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"
GLOBAL_MODEL_DIR = OUTPUT_ROOT / "models"

ORDINAL_ROOT = OUTPUT_ROOT / "ordinal_imbalance_uncertainty"
RUN_ROOT = ORDINAL_ROOT / f"run_{RUN_ID}"

PRED_DIR = RUN_ROOT / "predictions"
PROBA_DIR = RUN_ROOT / "probabilities"
METRIC_DIR = RUN_ROOT / "metrics"
PLOT_DIR = RUN_ROOT / "plots"
MODEL_DIR = RUN_ROOT / "models"
IMPORTANCE_DIR = RUN_ROOT / "feature_importance"
SPLIT_DIR = RUN_ROOT / "splits"
CALIBRATION_DIR = RUN_ROOT / "calibration"
ALLOCATION_INPUT_DIR = RUN_ROOT / "allocation_inputs"

for d in [
    DATA_ROOT,
    MODELING_DIR,
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    GLOBAL_MODEL_DIR,
    ORDINAL_ROOT,
    RUN_ROOT,
    PRED_DIR,
    PROBA_DIR,
    METRIC_DIR,
    PLOT_DIR,
    MODEL_DIR,
    IMPORTANCE_DIR,
    SPLIT_DIR,
    CALIBRATION_DIR,
    ALLOCATION_INPUT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_DATA_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet"

print("=" * 80)
print("AURORA-TWETF Ordinal, Imbalance-Aware, Uncertainty Models")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Run ID       :", RUN_ID)
print("Project root :", PUBLICATION_ROOT)
print("Input data   :", MODEL_DATA_PATH)
print("Run root     :", RUN_ROOT)
print("=" * 80)

# ============================================================
# 2. Configuration
# ============================================================

TARGET_COLS = [
    "TAIEX_regime_fixed_20d",
    "TAIEX_regime_fixed_60d",
]

CLASS_LABELS = [0, 1, 2, 3, 4]
N_CLASSES = len(CLASS_LABELS)

REGIME_LABEL_DEFINITION = {
    0: "Strong Bear",
    1: "Bear",
    2: "Neutral",
    3: "Bull",
    4: "Strong Bull",
}

# Same split as Notebook 03.
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

N_JOBS = -1

# Official deployable model zoo for Notebook 04.
# No previous-label persistence model is included.
MODEL_CONFIG = {
    # Ordinal cumulative-link style models:
    "O1_cumulative_logit_balanced": {
        "type": "cumulative_link",
        "base": "logistic_regression",
        "enabled": True,
        "description": "Ordinal cumulative binary decomposition with class-balanced logistic base models.",
    },
    "O2_cumulative_rf_balanced": {
        "type": "cumulative_link",
        "base": "random_forest",
        "enabled": True,
        "description": "Ordinal cumulative binary decomposition with random forest base models.",
    },
    "O3_cumulative_et_balanced": {
        "type": "cumulative_link",
        "base": "extra_trees",
        "enabled": True,
        "description": "Ordinal cumulative binary decomposition with extra trees base models.",
    },
    "O4_cumulative_lgbm_balanced": {
        "type": "cumulative_link",
        "base": "lightgbm",
        "enabled": HAS_LIGHTGBM,
        "description": "Ordinal cumulative binary decomposition with LightGBM base models.",
    },

    # Regression-to-ordinal models:
    "R1_ridge_regression_to_ordinal": {
        "type": "regression_to_ordinal",
        "base": "ridge",
        "enabled": True,
        "description": "Continuous class-score regression rounded to ordinal class.",
    },
    "R2_rf_regression_to_ordinal": {
        "type": "regression_to_ordinal",
        "base": "random_forest_regressor",
        "enabled": True,
        "description": "Random forest regressor mapped to ordinal class.",
    },
    "R3_et_regression_to_ordinal": {
        "type": "regression_to_ordinal",
        "base": "extra_trees_regressor",
        "enabled": True,
        "description": "Extra trees regressor mapped to ordinal class.",
    },
    "R4_lgbm_regression_to_ordinal": {
        "type": "regression_to_ordinal",
        "base": "lightgbm_regressor",
        "enabled": HAS_LIGHTGBM,
        "description": "LightGBM regressor mapped to ordinal class.",
    },
    "R5_xgb_regression_to_ordinal": {
        "type": "regression_to_ordinal",
        "base": "xgboost_regressor",
        "enabled": HAS_XGBOOST,
        "description": "XGBoost regressor mapped to ordinal class.",
    },

    # Calibrated multiclass baselines:
    "C1_calibrated_logistic_balanced": {
        "type": "calibrated_multiclass",
        "base": "logistic_regression",
        "enabled": True,
        "description": "Class-balanced logistic regression with validation-set calibration.",
    },
    "C2_calibrated_rf_balanced": {
        "type": "calibrated_multiclass",
        "base": "random_forest",
        "enabled": True,
        "description": "Class-balanced random forest with validation-set calibration.",
    },
    "C3_calibrated_lgbm_balanced": {
        "type": "calibrated_multiclass",
        "base": "lightgbm",
        "enabled": HAS_LIGHTGBM,
        "description": "Class-balanced LightGBM with validation-set calibration.",
    },
    "C4_calibrated_xgb_multiclass": {
        "type": "calibrated_multiclass",
        "base": "xgboost",
        "enabled": HAS_XGBOOST,
        "description": "XGBoost multiclass with validation-set calibration.",
    },

    # Ensemble:
    "E1_valid_model_probability_ensemble": {
        "type": "probability_ensemble",
        "base": "ensemble",
        "enabled": True,
        "description": "Validation-weighted probability ensemble over selected deployable models.",
    },
}

# Ensemble members must be model names defined above and trained before E1.
ENSEMBLE_CANDIDATES = [
    "O1_cumulative_logit_balanced",
    "O2_cumulative_rf_balanced",
    "O3_cumulative_et_balanced",
    "O4_cumulative_lgbm_balanced",
    "R1_ridge_regression_to_ordinal",
    "R2_rf_regression_to_ordinal",
    "R3_et_regression_to_ordinal",
    "R4_lgbm_regression_to_ordinal",
    "R5_xgb_regression_to_ordinal",
    "C1_calibrated_logistic_balanced",
    "C2_calibrated_rf_balanced",
    "C3_calibrated_lgbm_balanced",
    "C4_calibrated_xgb_multiclass",
]

# For uncertainty-aware allocation later, lower entropy and higher margin can be used as confidence signals.
EPS = 1e-12

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    path = Path(path)
    path.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return str(x).replace("/", "_").replace("\\", "_").replace(":", "_").replace(" ", "_")

def ordinal_mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def ordinal_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def adjacent_accuracy(y_true, y_pred, tolerance=1):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    return float(np.mean(np.abs(y_true - y_pred) <= tolerance))

def proba_to_pred(proba, labels=CLASS_LABELS):
    labels_arr = np.asarray(labels, dtype=int)
    return labels_arr[np.argmax(proba, axis=1)]

def expected_class_from_proba(proba, labels=CLASS_LABELS):
    labels_arr = np.asarray(labels, dtype=float)
    return proba @ labels_arr

def predictive_entropy(proba):
    proba = np.asarray(proba, dtype=float)
    return -np.sum(np.clip(proba, EPS, 1.0) * np.log(np.clip(proba, EPS, 1.0)), axis=1)

def normalized_entropy(proba):
    return predictive_entropy(proba) / np.log(proba.shape[1])

def probability_margin(proba):
    proba = np.asarray(proba, dtype=float)
    sorted_p = np.sort(proba, axis=1)
    return sorted_p[:, -1] - sorted_p[:, -2]

def ordinal_variance(proba, labels=CLASS_LABELS):
    proba = np.asarray(proba, dtype=float)
    labels_arr = np.asarray(labels, dtype=float)
    mu = proba @ labels_arr
    second = proba @ (labels_arr ** 2)
    return second - mu ** 2

def ensure_valid_proba(proba):
    proba = np.asarray(proba, dtype=float)
    proba = np.nan_to_num(proba, nan=0.0, posinf=0.0, neginf=0.0)
    proba[proba < 0] = 0.0
    row_sums = proba.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0
    if np.any(zero_rows):
        proba[zero_rows, :] = 1.0 / proba.shape[1]
        row_sums = proba.sum(axis=1, keepdims=True)
    return proba / row_sums

def align_proba_columns(proba, model_classes, all_classes=CLASS_LABELS):
    proba = np.asarray(proba, dtype=float)
    out = np.zeros((proba.shape[0], len(all_classes)), dtype=float)
    class_to_pos = {int(c): i for i, c in enumerate(model_classes)}
    for j, c in enumerate(all_classes):
        if int(c) in class_to_pos:
            out[:, j] = proba[:, class_to_pos[int(c)]]
    return ensure_valid_proba(out)

def multiclass_brier_score(y_true, proba, labels=CLASS_LABELS):
    y_true = np.asarray(y_true, dtype=int)
    proba = ensure_valid_proba(proba)
    y_onehot = np.zeros_like(proba)
    label_to_col = {c: i for i, c in enumerate(labels)}
    for i, y in enumerate(y_true):
        if int(y) in label_to_col:
            y_onehot[i, label_to_col[int(y)]] = 1.0
    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

def ece_multiclass(y_true, proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    proba = ensure_valid_proba(proba)
    y_pred = proba_to_pred(proba)
    conf = np.max(proba, axis=1)
    correct = (y_pred == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []

    for b in range(n_bins):
        lo, hi = bins[b], bins[b + 1]
        if b == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        n = int(mask.sum())
        if n == 0:
            rows.append({
                "bin": b,
                "confidence_low": lo,
                "confidence_high": hi,
                "n": 0,
                "avg_confidence": np.nan,
                "accuracy": np.nan,
                "abs_gap": np.nan,
            })
            continue

        avg_conf = float(conf[mask].mean())
        acc = float(correct[mask].mean())
        gap = abs(acc - avg_conf)
        ece += (n / len(y_true)) * gap

        rows.append({
            "bin": b,
            "confidence_low": lo,
            "confidence_high": hi,
            "n": n,
            "avg_confidence": avg_conf,
            "accuracy": acc,
            "abs_gap": gap,
        })

    return float(ece), pd.DataFrame(rows)

def evaluate_predictions(y_true, y_pred, proba=None, labels=CLASS_LABELS):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    metrics = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "quadratic_weighted_kappa": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "ordinal_mae": ordinal_mae(y_true, y_pred),
        "ordinal_rmse": ordinal_rmse(y_true, y_pred),
        "adjacent_accuracy_tol_1": adjacent_accuracy(y_true, y_pred, tolerance=1),
        "large_error_rate_abs_ge_2": float(np.mean(np.abs(y_true - y_pred) >= 2)),
        "extreme_error_rate_abs_ge_3": float(np.mean(np.abs(y_true - y_pred) >= 3)),
    }

    if proba is not None:
        proba = ensure_valid_proba(proba)
        try:
            metrics["multiclass_log_loss"] = float(log_loss(y_true, proba, labels=labels))
        except Exception:
            metrics["multiclass_log_loss"] = np.nan

        metrics["multiclass_brier"] = multiclass_brier_score(y_true, proba, labels=labels)
        ece, _ = ece_multiclass(y_true, proba, n_bins=10)
        metrics["ece_10bin"] = ece

        exp_class = expected_class_from_proba(proba, labels=labels)
        metrics["expected_class_mae"] = float(np.mean(np.abs(y_true - exp_class)))
        metrics["expected_class_rmse"] = float(np.sqrt(np.mean((y_true - exp_class) ** 2)))
        metrics["mean_max_probability"] = float(np.max(proba, axis=1).mean())
        metrics["mean_entropy"] = float(predictive_entropy(proba).mean())
        metrics["mean_normalized_entropy"] = float(normalized_entropy(proba).mean())
        metrics["mean_probability_margin"] = float(probability_margin(proba).mean())
        metrics["mean_ordinal_variance"] = float(ordinal_variance(proba, labels=labels).mean())
    else:
        metrics["multiclass_log_loss"] = np.nan
        metrics["multiclass_brier"] = np.nan
        metrics["ece_10bin"] = np.nan
        metrics["expected_class_mae"] = np.nan
        metrics["expected_class_rmse"] = np.nan
        metrics["mean_max_probability"] = np.nan
        metrics["mean_entropy"] = np.nan
        metrics["mean_normalized_entropy"] = np.nan
        metrics["mean_probability_margin"] = np.nan
        metrics["mean_ordinal_variance"] = np.nan

    return metrics

def make_classification_report_df(y_true, y_pred, labels=CLASS_LABELS):
    target_names = [REGIME_LABEL_DEFINITION[c] for c in labels]
    rep = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(rep).T.reset_index().rename(columns={"index": "class_or_average"})

def plot_confusion_matrix(cm, title, path, labels=CLASS_LABELS, normalize=False):
    plt.figure(figsize=(7.5, 6))

    if normalize:
        cm_plot = cm.astype(float)
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        cm_plot = cm_plot / row_sums
        fmt = ".2f"
    else:
        cm_plot = cm
        fmt = "d"

    tick_labels = [f"{c}\n{REGIME_LABEL_DEFINITION[c]}" for c in labels]

    sns.heatmap(
        cm_plot,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=tick_labels,
        yticklabels=tick_labels,
        cbar=True,
    )

    plt.title(title)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_metric_comparison(metric_df, target_col, split_name, metric_name, path, higher_is_better=True):
    dfp = metric_df[
        (metric_df["target_col"] == target_col)
        & (metric_df["split"] == split_name)
    ].copy()

    if dfp.empty:
        return

    dfp = dfp.sort_values(metric_name, ascending=not higher_is_better)

    plt.figure(figsize=(10, max(4, 0.45 * len(dfp))))
    sns.barplot(data=dfp, y="model_name", x=metric_name, color="#55A868")
    direction = "higher is better" if higher_is_better else "lower is better"
    plt.title(f"{target_col} | {split_name} | {metric_name} ({direction})")
    plt.xlabel(metric_name)
    plt.ylabel("Model")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_reliability_curve(cal_df, title, path):
    dfp = cal_df.dropna(subset=["avg_confidence", "accuracy"]).copy()

    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")

    if not dfp.empty:
        plt.plot(
            dfp["avg_confidence"],
            dfp["accuracy"],
            marker="o",
            color="#4C72B0",
            label="Observed",
        )

    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel("Mean predicted confidence")
    plt.ylabel("Empirical accuracy")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_uncertainty_error_relation(pred_df, title, path):
    cols = ["normalized_entropy", "abs_class_error"]
    if not all(c in pred_df.columns for c in cols):
        return

    dfp = pred_df.dropna(subset=cols).copy()
    if dfp.empty:
        return

    dfp["entropy_bin"] = pd.qcut(
        dfp["normalized_entropy"],
        q=min(5, dfp["normalized_entropy"].nunique()),
        duplicates="drop",
    )

    agg = dfp.groupby("entropy_bin", observed=True).agg(
        mean_entropy=("normalized_entropy", "mean"),
        mean_abs_error=("abs_class_error", "mean"),
        n=("abs_class_error", "size"),
    ).reset_index()

    plt.figure(figsize=(7, 4.5))
    sns.lineplot(data=agg, x="mean_entropy", y="mean_abs_error", marker="o")
    for _, row in agg.iterrows():
        plt.text(row["mean_entropy"], row["mean_abs_error"], f"n={int(row['n'])}", fontsize=8)
    plt.xlabel("Mean normalized entropy")
    plt.ylabel("Mean absolute class error")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_probability_heatmap(proba_df, title, path):
    proba_cols = [f"proba_class_{c}" for c in CLASS_LABELS]
    if not all(c in proba_df.columns for c in proba_cols):
        return

    dfp = proba_df.sort_index().copy()
    mat = dfp[proba_cols].T.values

    plt.figure(figsize=(12, 4.5))
    sns.heatmap(mat, cmap="viridis", cbar=True)
    plt.yticks(
        ticks=np.arange(len(CLASS_LABELS)) + 0.5,
        labels=[f"{c}: {REGIME_LABEL_DEFINITION[c]}" for c in CLASS_LABELS],
        rotation=0,
    )
    plt.xlabel("Time-ordered observations")
    plt.ylabel("Class probability")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def plot_feature_importance(fi_df, title, path, top_n=30):
    if fi_df is None or fi_df.empty:
        return

    dfp = fi_df.sort_values("importance", ascending=False).head(top_n).copy()

    plt.figure(figsize=(10, max(5, 0.32 * len(dfp))))
    sns.barplot(data=dfp, x="importance", y="feature", color="#C44E52")
    plt.title(title)
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

# ============================================================
# 4. Ordinal helper classes
# ============================================================

class CumulativeOrdinalClassifier(BaseEstimator, ClassifierMixin):
    """
    Ordinal classifier using cumulative binary decomposition.

    For K ordered classes, fit K-1 binary models:
        model_k estimates P(Y > k) for k = 0, 1, ..., K-2.

    Class probabilities are recovered as:
        P(Y = 0)     = 1 - P(Y > 0)
        P(Y = j)     = P(Y > j-1) - P(Y > j), 1 <= j <= K-2
        P(Y = K - 1) = P(Y > K-2)

    The cumulative probabilities are monotonized to avoid invalid negative class probabilities.
    """

    def __init__(self, base_estimator, labels=None, monotonic_repair=True):
        self.base_estimator = base_estimator
        self.labels = CLASS_LABELS if labels is None else list(labels)
        self.monotonic_repair = monotonic_repair

    def fit(self, X, y):
        y = np.asarray(y, dtype=int)
        self.classes_ = np.asarray(self.labels, dtype=int)
        self.thresholds_ = self.labels[:-1]
        self.models_ = []

        for threshold in self.thresholds_:
            binary_y = (y > threshold).astype(int)
            model = clone(self.base_estimator)
            model.fit(X, binary_y)
            self.models_.append(model)

        return self

    def _predict_cumulative_greater(self, X):
        cumulative = []
        for model in self.models_:
            if hasattr(model, "predict_proba"):
                p = model.predict_proba(X)
                classes = getattr(model, "classes_", np.array([0, 1]))
                if 1 in classes:
                    p_gt = p[:, list(classes).index(1)]
                else:
                    p_gt = np.zeros(X.shape[0])
            else:
                pred = model.predict(X)
                p_gt = np.asarray(pred, dtype=float)
            cumulative.append(p_gt)

        cumulative = np.vstack(cumulative).T

        if self.monotonic_repair and cumulative.shape[1] > 1:
            # Enforce P(Y>0) >= P(Y>1) >= ... >= P(Y>K-2).
            cumulative = np.minimum.accumulate(cumulative, axis=1)

        cumulative = np.clip(cumulative, 0.0, 1.0)
        return cumulative

    def predict_proba(self, X):
        cumulative = self._predict_cumulative_greater(X)
        n = cumulative.shape[0]
        k = len(self.labels)
        proba = np.zeros((n, k), dtype=float)

        proba[:, 0] = 1.0 - cumulative[:, 0]

        for j in range(1, k - 1):
            proba[:, j] = cumulative[:, j - 1] - cumulative[:, j]

        proba[:, k - 1] = cumulative[:, k - 2]

        return ensure_valid_proba(proba)

    def predict(self, X):
        proba = self.predict_proba(X)
        return proba_to_pred(proba, labels=self.labels)

class RegressionToOrdinalClassifier(BaseEstimator, ClassifierMixin):
    """
    Fits a regression model on ordinal class labels.
    Converts continuous score to class probabilities using distance-based softmax.

    This is deployable because it uses only X features, not future labels.
    """

    def __init__(self, regressor, labels=None, temperature=0.70):
        self.regressor = regressor
        self.labels = CLASS_LABELS if labels is None else list(labels)
        self.temperature = float(temperature)

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.classes_ = np.asarray(self.labels, dtype=int)
        self.model_ = clone(self.regressor)
        self.model_.fit(X, y)
        return self

    def predict_score(self, X):
        score = np.asarray(self.model_.predict(X), dtype=float)
        return np.clip(score, min(self.labels), max(self.labels))

    def predict_proba(self, X):
        score = self.predict_score(X)
        labels_arr = np.asarray(self.labels, dtype=float)
        dist2 = (score[:, None] - labels_arr[None, :]) ** 2
        logits = -dist2 / max(self.temperature, 1e-6)
        logits = logits - logits.max(axis=1, keepdims=True)
        proba = np.exp(logits)
        return ensure_valid_proba(proba)

    def predict(self, X):
        score = self.predict_score(X)
        pred = np.rint(score).astype(int)
        pred = np.clip(pred, min(self.labels), max(self.labels))
        return pred

class ValidationTemperatureScaler:
    """
    Simple probability temperature scaling.
    Fits temperature on validation set by minimizing log loss.
    """

    def __init__(self, temperatures=None):
        if temperatures is None:
            temperatures = np.linspace(0.50, 5.00, 91)
        self.temperatures = np.asarray(temperatures, dtype=float)

    def fit(self, proba_val, y_val, labels=CLASS_LABELS):
        proba_val = ensure_valid_proba(proba_val)
        y_val = np.asarray(y_val, dtype=int)

        best_temp = 1.0
        best_loss = np.inf

        logp = np.log(np.clip(proba_val, EPS, 1.0))

        for temp in self.temperatures:
            scaled_logits = logp / temp
            scaled_logits = scaled_logits - scaled_logits.max(axis=1, keepdims=True)
            scaled = np.exp(scaled_logits)
            scaled = ensure_valid_proba(scaled)

            try:
                loss = log_loss(y_val, scaled, labels=labels)
            except Exception:
                loss = np.inf

            if loss < best_loss:
                best_loss = loss
                best_temp = temp

        self.temperature_ = float(best_temp)
        self.validation_log_loss_ = float(best_loss)
        return self

    def transform(self, proba):
        proba = ensure_valid_proba(proba)
        logp = np.log(np.clip(proba, EPS, 1.0))
        scaled_logits = logp / self.temperature_
        scaled_logits = scaled_logits - scaled_logits.max(axis=1, keepdims=True)
        scaled = np.exp(scaled_logits)
        return ensure_valid_proba(scaled)

class ProbabilityEnsemble:
    """
    Weighted average probability ensemble.
    """

    def __init__(self, model_names, weights):
        self.model_names = list(model_names)
        self.weights = np.asarray(weights, dtype=float)
        if self.weights.sum() <= 0:
            self.weights = np.ones(len(self.model_names)) / len(self.model_names)
        else:
            self.weights = self.weights / self.weights.sum()

    def combine(self, proba_dict):
        probas = []
        used_weights = []
        for name, weight in zip(self.model_names, self.weights):
            if name in proba_dict:
                probas.append(ensure_valid_proba(proba_dict[name]))
                used_weights.append(weight)

        if not probas:
            raise ValueError("No probability matrices available for ensemble.")

        used_weights = np.asarray(used_weights, dtype=float)
        used_weights = used_weights / used_weights.sum()

        out = np.zeros_like(probas[0])
        for p, w in zip(probas, used_weights):
            out += w * p

        return ensure_valid_proba(out)

# ============================================================
# 5. Model factory
# ============================================================

def make_binary_base_estimator(base_name):
    if base_name == "logistic_regression":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=0.50,
                solver="lbfgs",
                class_weight="balanced",
                max_iter=3000,
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if base_name == "random_forest":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=400,
                max_depth=5,
                min_samples_leaf=20,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if base_name == "extra_trees":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=400,
                max_depth=5,
                min_samples_leaf=20,
                max_features="sqrt",
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if base_name == "lightgbm":
        if not HAS_LIGHTGBM:
            raise RuntimeError("LightGBM is not available.")
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", LGBMClassifier(
                objective="binary",
                n_estimators=300,
                learning_rate=0.025,
                num_leaves=7,
                max_depth=3,
                min_child_samples=30,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.30,
                reg_lambda=0.80,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbose=-1,
            )),
        ])

    raise ValueError(f"Unknown binary base estimator: {base_name}")

def make_multiclass_base_estimator(base_name):
    if base_name == "logistic_regression":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=0.50,
                solver="lbfgs",
                multi_class="auto",
                class_weight="balanced",
                max_iter=3000,
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if base_name == "random_forest":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=500,
                max_depth=6,
                min_samples_leaf=20,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if base_name == "lightgbm":
        if not HAS_LIGHTGBM:
            raise RuntimeError("LightGBM is not available.")
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", LGBMClassifier(
                objective="multiclass",
                num_class=N_CLASSES,
                n_estimators=350,
                learning_rate=0.025,
                num_leaves=7,
                max_depth=3,
                min_child_samples=30,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.30,
                reg_lambda=0.80,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbose=-1,
            )),
        ])

    if base_name == "xgboost":
        if not HAS_XGBOOST:
            raise RuntimeError("XGBoost is not available.")
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                objective="multi:softprob",
                num_class=N_CLASSES,
                n_estimators=300,
                learning_rate=0.025,
                max_depth=2,
                min_child_weight=5,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.30,
                reg_lambda=1.50,
                eval_metric="mlogloss",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbosity=0,
            )),
        ])

    raise ValueError(f"Unknown multiclass base estimator: {base_name}")

def make_regressor(base_name):
    if base_name == "ridge":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("reg", Ridge(
                alpha=10.0,
                random_state=RANDOM_SEED,
            )),
        ])

    if base_name == "random_forest_regressor":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("reg", RandomForestRegressor(
                n_estimators=400,
                max_depth=5,
                min_samples_leaf=20,
                max_features="sqrt",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if base_name == "extra_trees_regressor":
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("reg", ExtraTreesRegressor(
                n_estimators=400,
                max_depth=5,
                min_samples_leaf=20,
                max_features="sqrt",
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
            )),
        ])

    if base_name == "lightgbm_regressor":
        if not HAS_LIGHTGBM:
            raise RuntimeError("LightGBM is not available.")
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("reg", LGBMRegressor(
                objective="regression",
                n_estimators=350,
                learning_rate=0.025,
                num_leaves=7,
                max_depth=3,
                min_child_samples=30,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.30,
                reg_lambda=0.80,
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbose=-1,
            )),
        ])

    if base_name == "xgboost_regressor":
        if not HAS_XGBOOST:
            raise RuntimeError("XGBoost is not available.")
        return Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("reg", XGBRegressor(
                objective="reg:squarederror",
                n_estimators=300,
                learning_rate=0.025,
                max_depth=2,
                min_child_weight=5,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_alpha=0.30,
                reg_lambda=1.50,
                random_state=RANDOM_SEED,
                n_jobs=N_JOBS,
                verbosity=0,
            )),
        ])

    raise ValueError(f"Unknown regressor: {base_name}")

def build_model(model_type, base_name):
    if model_type == "cumulative_link":
        base = make_binary_base_estimator(base_name)
        return CumulativeOrdinalClassifier(base_estimator=base, labels=CLASS_LABELS)

    if model_type == "regression_to_ordinal":
        reg = make_regressor(base_name)
        return RegressionToOrdinalClassifier(regressor=reg, labels=CLASS_LABELS, temperature=0.70)

    if model_type == "calibrated_multiclass":
        return make_multiclass_base_estimator(base_name)

    if model_type == "probability_ensemble":
        return None

    raise ValueError(f"Unknown model_type: {model_type}")

def get_model_classes(model):
    if hasattr(model, "classes_"):
        return model.classes_
    if isinstance(model, Pipeline):
        clf = model.named_steps.get("clf", None)
        if clf is not None and hasattr(clf, "classes_"):
            return clf.classes_
    return np.asarray(CLASS_LABELS, dtype=int)

def get_feature_importance(model, feature_cols):
    """
    Best-effort feature importance extraction.
    """
    rows = []

    # CumulativeOrdinalClassifier: average importances across binary threshold models.
    if isinstance(model, CumulativeOrdinalClassifier):
        fi_list = []
        for threshold, submodel in zip(model.thresholds_, model.models_):
            sub_fi = get_feature_importance(submodel, feature_cols)
            if sub_fi is not None and not sub_fi.empty:
                sub_fi = sub_fi.copy()
                sub_fi["threshold"] = threshold
                fi_list.append(sub_fi)

        if fi_list:
            tmp = pd.concat(fi_list, ignore_index=True)
            agg = tmp.groupby("feature", as_index=False)["importance"].mean()
            return agg.sort_values("importance", ascending=False)

    # RegressionToOrdinalClassifier.
    if isinstance(model, RegressionToOrdinalClassifier):
        return get_feature_importance(model.model_, feature_cols)

    # Pipeline.
    if isinstance(model, Pipeline):
        estimator = None
        for key in ["clf", "reg"]:
            if key in model.named_steps:
                estimator = model.named_steps[key]
                break

        if estimator is not None:
            if hasattr(estimator, "feature_importances_"):
                imp = np.asarray(estimator.feature_importances_, dtype=float)
                return pd.DataFrame({
                    "feature": feature_cols,
                    "importance": imp,
                }).sort_values("importance", ascending=False)

            if hasattr(estimator, "coef_"):
                coef = np.asarray(estimator.coef_, dtype=float)
                if coef.ndim == 2:
                    imp = np.mean(np.abs(coef), axis=0)
                else:
                    imp = np.abs(coef)
                return pd.DataFrame({
                    "feature": feature_cols,
                    "importance": imp,
                }).sort_values("importance", ascending=False)

    # Direct estimator.
    if hasattr(model, "feature_importances_"):
        imp = np.asarray(model.feature_importances_, dtype=float)
        return pd.DataFrame({
            "feature": feature_cols,
            "importance": imp,
        }).sort_values("importance", ascending=False)

    if hasattr(model, "coef_"):
        coef = np.asarray(model.coef_, dtype=float)
        if coef.ndim == 2:
            imp = np.mean(np.abs(coef), axis=0)
        else:
            imp = np.abs(coef)
        return pd.DataFrame({
            "feature": feature_cols,
            "importance": imp,
        }).sort_values("importance", ascending=False)

    return pd.DataFrame(columns=["feature", "importance"])

# ============================================================
# 6. Valid trailing-regime proxy features
# ============================================================

def make_realized_return_regime(realized_return):
    """
    Same threshold rule as target labels, but applied only to trailing realized return.
    This is valid because it uses past information.
    """
    r = pd.Series(realized_return).copy()
    out = pd.Series(index=r.index, dtype="float")

    out[r < -0.10] = 0
    out[(r >= -0.10) & (r < -0.03)] = 1
    out[(r >= -0.03) & (r <= 0.03)] = 2
    out[(r > 0.03) & (r <= 0.10)] = 3
    out[r > 0.10] = 4

    return out

def add_valid_trailing_regime_features(df):
    """
    Adds valid regime-persistence-like features based on trailing realized returns.
    Does NOT use previous target labels.
    """
    out = df.copy()

    # Try to find a TAIEX close column.
    possible_close_cols = [
        "TAIEX_close",
        "close_TAIEX",
        "TAIEX__close",
        "TAIEX",
    ]

    close_col = None
    for c in possible_close_cols:
        if c in out.columns:
            close_col = c
            break

    # If no close column is available, try feature names containing TAIEX and close.
    if close_col is None:
        candidates = [
            c for c in out.columns
            if ("TAIEX" in str(c).upper()) and ("CLOSE" in str(c).upper())
        ]
        if candidates:
            close_col = candidates[0]

    if close_col is None:
        print("No TAIEX close column found. Skipping trailing realized regime features.")
        return out, []

    new_cols = []

    for h in [20, 60]:
        ret_col = f"TAIEX_trailing_return_{h}d_valid"
        reg_col = f"TAIEX_trailing_regime_{h}d_valid"
        abs_col = f"TAIEX_trailing_return_abs_{h}d_valid"

        out[ret_col] = out[close_col].pct_change(h)
        out[reg_col] = make_realized_return_regime(out[ret_col])
        out[abs_col] = out[ret_col].abs()

        new_cols.extend([ret_col, reg_col, abs_col])

    print(f"Added valid trailing-regime features using close column: {close_col}")
    print("New columns:", new_cols)

    return out, new_cols

# ============================================================
# 7. Load modeling dataset
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading modeling dataset")
print("=" * 80)

if not MODEL_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing modeling dataset: {MODEL_DATA_PATH}\n"
        "Please run 02_AURORA_data_preparation_leakage_controlled.ipynb first."
    )

df = pd.read_parquet(MODEL_DATA_PATH)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

print("Original dataset shape:", df.shape)
print("Original date range   :", df.index.min().date(), "to", df.index.max().date())

for target_col in TARGET_COLS:
    if target_col not in df.columns:
        raise ValueError(f"Missing target column: {target_col}")
    df[target_col] = df[target_col].astype(int)

# Add valid trailing realized regime features if possible.
df, trailing_feature_cols = add_valid_trailing_regime_features(df)

# Exclude future/audit/target columns.
EXCLUDE_KEYWORDS = [
    "future",
    "audit",
    "target",
    "regime_fixed",
]

feature_cols = []
for c in df.columns:
    if c in TARGET_COLS:
        continue
    c_lower = str(c).lower()
    if any(k in c_lower for k in EXCLUDE_KEYWORDS):
        continue
    feature_cols.append(c)

# Remove rows with missing added trailing features if those were added.
# This is small and avoids invalid imputation of regime proxy warm-up.
if trailing_feature_cols:
    before = len(df)
    df = df.dropna(subset=trailing_feature_cols).copy()
    after = len(df)
    print(f"Dropped {before - after} rows due to trailing valid regime feature warm-up.")
else:
    print("No trailing feature warm-up rows dropped.")

print("Final loaded dataset shape:", df.shape)
print("Final date range          :", df.index.min().date(), "to", df.index.max().date())
print("Feature columns           :", len(feature_cols))
print("Target columns            :", TARGET_COLS)

dataset_summary = {
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "input_path": str(MODEL_DATA_PATH),
    "dataset_shape_after_optional_trailing_features": df.shape,
    "date_start": str(df.index.min().date()),
    "date_end": str(df.index.max().date()),
    "n_features": len(feature_cols),
    "target_cols": TARGET_COLS,
    "trailing_feature_cols": trailing_feature_cols,
    "note": "No previous target labels are used as features.",
}
save_json(RUN_ROOT / "dataset_summary.json", dataset_summary)
save_json(REPORT_DIR / f"AURORA_04_dataset_summary_{RUN_ID}.json", dataset_summary)

# ============================================================
# 8. Chronological train / validation / test split
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Chronological train / validation / test split")
print("=" * 80)

n = len(df)
train_end = int(np.floor(n * TRAIN_FRAC))
val_end = int(np.floor(n * (TRAIN_FRAC + VAL_FRAC)))

train_idx = df.index[:train_end]
val_idx = df.index[train_end:val_end]
test_idx = df.index[val_end:]

split_report = pd.DataFrame([
    {
        "split": "train",
        "n": len(train_idx),
        "start_date": train_idx.min().date(),
        "end_date": train_idx.max().date(),
    },
    {
        "split": "validation",
        "n": len(val_idx),
        "start_date": val_idx.min().date(),
        "end_date": val_idx.max().date(),
    },
    {
        "split": "test",
        "n": len(test_idx),
        "start_date": test_idx.min().date(),
        "end_date": test_idx.max().date(),
    },
])

print(split_report.to_string(index=False))

split_report.to_csv(TABLE_DIR / f"table_16_ordinal_split_report_{RUN_ID}.csv", index=False)
split_report.to_csv(RUN_ROOT / "split_report.csv", index=False)

split_assignment = pd.DataFrame(index=df.index)
split_assignment["split"] = "unused"
split_assignment.loc[train_idx, "split"] = "train"
split_assignment.loc[val_idx, "split"] = "validation"
split_assignment.loc[test_idx, "split"] = "test"
split_assignment.to_csv(SPLIT_DIR / "row_split_assignment.csv")

# ============================================================
# 9. Training and evaluation
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Training ordinal, imbalance-aware, uncertainty models")
print("=" * 80)

X_all = df[feature_cols].copy()

all_metric_rows = []
all_class_report_rows = []
all_prediction_frames = []
all_probability_frames = []
all_feature_importance_frames = []
all_calibration_rows = []
all_model_summary_rows = []

# Store per target/model probabilities for ensemble.
stored_probas = {}
stored_models = {}

for target_col in TARGET_COLS:
    print("\n" + "-" * 80)
    print(f"Target: {target_col}")
    print("-" * 80)

    y_all = df[target_col].astype(int).copy()

    X_train = X_all.loc[train_idx]
    X_val = X_all.loc[val_idx]
    X_test = X_all.loc[test_idx]

    y_train = y_all.loc[train_idx]
    y_val = y_all.loc[val_idx]
    y_test = y_all.loc[test_idx]

    split_objects = {
        "train": (X_train, y_train, train_idx),
        "validation": (X_val, y_val, val_idx),
        "test": (X_test, y_test, test_idx),
    }

    stored_probas[target_col] = {}
    stored_models[target_col] = {}

    target_distribution_rows = []
    for split_name, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
        counts = y_all.loc[idx].value_counts().reindex(CLASS_LABELS, fill_value=0)
        shares = counts / counts.sum()
        for c in CLASS_LABELS:
            target_distribution_rows.append({
                "target_col": target_col,
                "split": split_name,
                "class": c,
                "regime_name": REGIME_LABEL_DEFINITION[c],
                "n": int(counts.loc[c]),
                "share": float(shares.loc[c]),
            })

    pd.DataFrame(target_distribution_rows).to_csv(
        TABLE_DIR / f"table_17_ordinal_target_split_distribution_{safe_name(target_col)}_{RUN_ID}.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Train non-ensemble models first.
    # --------------------------------------------------------
    for model_name, cfg in MODEL_CONFIG.items():
        if not cfg.get("enabled", True):
            print(f"Skipping disabled model: {model_name}")
            continue

        if cfg["type"] == "probability_ensemble":
            continue

        model_type = cfg["type"]
        base_name = cfg["base"]

        print(f"\nTraining model: {model_name}")
        print(f"Description   : {cfg.get('description', '')}")

        start_time = time.time()

        try:
            model = build_model(model_type, base_name)
            model.fit(X_train, y_train)

            raw_train_proba = ensure_valid_proba(model.predict_proba(X_train))
            raw_val_proba = ensure_valid_proba(model.predict_proba(X_val))
            raw_test_proba = ensure_valid_proba(model.predict_proba(X_test))

            # Temperature scaling fitted only on validation.
            temp_scaler = ValidationTemperatureScaler()
            temp_scaler.fit(raw_val_proba, y_val.values, labels=CLASS_LABELS)

            train_proba = temp_scaler.transform(raw_train_proba)
            val_proba = temp_scaler.transform(raw_val_proba)
            test_proba = temp_scaler.transform(raw_test_proba)

            fit_seconds = time.time() - start_time

            model_artifact = {
                "model": model,
                "temperature_scaler": temp_scaler,
                "feature_cols": feature_cols,
                "target_col": target_col,
                "model_name": model_name,
                "model_config": cfg,
            }

            model_path = MODEL_DIR / f"model_{safe_name(target_col)}_{model_name}.joblib"
            joblib.dump(model_artifact, model_path)

            stored_models[target_col][model_name] = model_artifact
            stored_probas[target_col][model_name] = {
                "train": train_proba,
                "validation": val_proba,
                "test": test_proba,
            }

            all_model_summary_rows.append({
                "run_id": RUN_ID,
                "target_col": target_col,
                "model_name": model_name,
                "model_type": model_type,
                "base_name": base_name,
                "status": "success",
                "fit_seconds": float(fit_seconds),
                "temperature": float(temp_scaler.temperature_),
                "validation_log_loss_after_temperature": float(temp_scaler.validation_log_loss_),
                "model_path": str(model_path),
            })

            # Evaluate splits.
            split_proba_map = {
                "train": train_proba,
                "validation": val_proba,
                "test": test_proba,
            }

            model_prediction_frames = []
            model_probability_frames = []

            for split_name, (X_split, y_split, idx_split) in split_objects.items():
                proba = split_proba_map[split_name]
                y_pred = proba_to_pred(proba, labels=CLASS_LABELS)

                metric_values = evaluate_predictions(
                    y_true=y_split.values,
                    y_pred=y_pred,
                    proba=proba,
                    labels=CLASS_LABELS,
                )

                metric_row = {
                    "run_id": RUN_ID,
                    "run_timestamp_utc": RUN_TIMESTAMP,
                    "target_col": target_col,
                    "model_name": model_name,
                    "model_type": model_type,
                    "base_name": base_name,
                    "split": split_name,
                    "fit_seconds": float(fit_seconds),
                    "temperature": float(temp_scaler.temperature_),
                    **metric_values,
                }

                all_metric_rows.append(metric_row)

                print(
                    f"{split_name:>10} | "
                    f"acc={metric_values['accuracy']:.4f} | "
                    f"bal_acc={metric_values['balanced_accuracy']:.4f} | "
                    f"macro_f1={metric_values['macro_f1']:.4f} | "
                    f"ord_mae={metric_values['ordinal_mae']:.4f} | "
                    f"qwk={metric_values['quadratic_weighted_kappa']:.4f} | "
                    f"ece={metric_values['ece_10bin']:.4f}"
                )

                # Class report.
                cr_df = make_classification_report_df(
                    y_true=y_split.values,
                    y_pred=y_pred,
                    labels=CLASS_LABELS,
                )
                cr_df.insert(0, "split", split_name)
                cr_df.insert(0, "base_name", base_name)
                cr_df.insert(0, "model_type", model_type)
                cr_df.insert(0, "model_name", model_name)
                cr_df.insert(0, "target_col", target_col)
                cr_df.insert(0, "run_id", RUN_ID)
                all_class_report_rows.append(cr_df)

                # Confusion matrices.
                cm = confusion_matrix(y_split.values, y_pred, labels=CLASS_LABELS)
                cm_df = pd.DataFrame(
                    cm,
                    index=[f"true_{c}" for c in CLASS_LABELS],
                    columns=[f"pred_{c}" for c in CLASS_LABELS],
                )
                cm_df.to_csv(
                    METRIC_DIR / f"confusion_matrix_{safe_name(target_col)}_{model_name}_{split_name}.csv"
                )

                plot_confusion_matrix(
                    cm,
                    title=f"{target_col} | {model_name} | {split_name}",
                    path=PLOT_DIR / f"confusion_matrix_{safe_name(target_col)}_{model_name}_{split_name}.png",
                    labels=CLASS_LABELS,
                    normalize=False,
                )

                plot_confusion_matrix(
                    cm,
                    title=f"{target_col} | {model_name} | {split_name} | normalized",
                    path=PLOT_DIR / f"confusion_matrix_normalized_{safe_name(target_col)}_{model_name}_{split_name}.png",
                    labels=CLASS_LABELS,
                    normalize=True,
                )

                # Calibration diagnostics.
                ece, cal_df = ece_multiclass(y_split.values, proba, n_bins=10)
                cal_df.insert(0, "split", split_name)
                cal_df.insert(0, "base_name", base_name)
                cal_df.insert(0, "model_type", model_type)
                cal_df.insert(0, "model_name", model_name)
                cal_df.insert(0, "target_col", target_col)
                cal_df.insert(0, "run_id", RUN_ID)
                all_calibration_rows.append(cal_df)

                cal_df.to_csv(
                    CALIBRATION_DIR / f"calibration_{safe_name(target_col)}_{model_name}_{split_name}.csv",
                    index=False,
                )

                plot_reliability_curve(
                    cal_df,
                    title=f"{target_col} | {model_name} | {split_name} | reliability",
                    path=PLOT_DIR / f"reliability_{safe_name(target_col)}_{model_name}_{split_name}.png",
                )

                # Prediction frame.
                pred_df = pd.DataFrame(index=idx_split)
                pred_df.index.name = "date"
                pred_df["run_id"] = RUN_ID
                pred_df["target_col"] = target_col
                pred_df["model_name"] = model_name
                pred_df["model_type"] = model_type
                pred_df["base_name"] = base_name
                pred_df["split"] = split_name
                pred_df["y_true"] = y_split.values.astype(int)
                pred_df["y_pred"] = y_pred.astype(int)
                pred_df["abs_class_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])
                pred_df["squared_class_error"] = (pred_df["y_true"] - pred_df["y_pred"]) ** 2
                pred_df["expected_class"] = expected_class_from_proba(proba, labels=CLASS_LABELS)
                pred_df["expected_class_error"] = pred_df["y_true"] - pred_df["expected_class"]
                pred_df["max_probability"] = np.max(proba, axis=1)
                pred_df["entropy"] = predictive_entropy(proba)
                pred_df["normalized_entropy"] = normalized_entropy(proba)
                pred_df["probability_margin"] = probability_margin(proba)
                pred_df["ordinal_variance"] = ordinal_variance(proba, labels=CLASS_LABELS)
                pred_df["confidence_score"] = 1.0 - pred_df["normalized_entropy"]
                pred_df["is_correct"] = (pred_df["y_true"] == pred_df["y_pred"]).astype(int)
                pred_df["is_adjacent_or_correct"] = (pred_df["abs_class_error"] <= 1).astype(int)

                model_prediction_frames.append(pred_df)
                all_prediction_frames.append(pred_df)

                # Probability frame.
                proba_df = pd.DataFrame(
                    proba,
                    index=idx_split,
                    columns=[f"proba_class_{c}" for c in CLASS_LABELS],
                )
                proba_df.index.name = "date"
                proba_df.insert(0, "split", split_name)
                proba_df.insert(0, "base_name", base_name)
                proba_df.insert(0, "model_type", model_type)
                proba_df.insert(0, "model_name", model_name)
                proba_df.insert(0, "target_col", target_col)
                proba_df.insert(0, "run_id", RUN_ID)

                model_probability_frames.append(proba_df)
                all_probability_frames.append(proba_df)

                if split_name == "test":
                    plot_uncertainty_error_relation(
                        pred_df,
                        title=f"{target_col} | {model_name} | uncertainty vs error",
                        path=PLOT_DIR / f"uncertainty_error_{safe_name(target_col)}_{model_name}_test.png",
                    )

                    plot_probability_heatmap(
                        proba_df,
                        title=f"{target_col} | {model_name} | test probability heatmap",
                        path=PLOT_DIR / f"probability_heatmap_{safe_name(target_col)}_{model_name}_test.png",
                    )

            # Save per-model predictions and probabilities.
            pd.concat(model_prediction_frames, axis=0).to_parquet(
                PRED_DIR / f"predictions_{safe_name(target_col)}_{model_name}.parquet"
            )
            pd.concat(model_prediction_frames, axis=0).to_csv(
                PRED_DIR / f"predictions_{safe_name(target_col)}_{model_name}.csv"
            )

            pd.concat(model_probability_frames, axis=0).to_parquet(
                PROBA_DIR / f"probabilities_{safe_name(target_col)}_{model_name}.parquet"
            )
            pd.concat(model_probability_frames, axis=0).to_csv(
                PROBA_DIR / f"probabilities_{safe_name(target_col)}_{model_name}.csv"
            )

            # Feature importance.
            fi_df = get_feature_importance(model, feature_cols)
            if fi_df is not None and not fi_df.empty:
                fi_df.insert(0, "base_name", base_name)
                fi_df.insert(0, "model_type", model_type)
                fi_df.insert(0, "model_name", model_name)
                fi_df.insert(0, "target_col", target_col)
                fi_df.insert(0, "run_id", RUN_ID)

                fi_df.to_csv(
                    IMPORTANCE_DIR / f"feature_importance_{safe_name(target_col)}_{model_name}.csv",
                    index=False,
                )
                all_feature_importance_frames.append(fi_df)

                plot_feature_importance(
                    fi_df,
                    title=f"{target_col} | {model_name} | top feature importances",
                    path=PLOT_DIR / f"feature_importance_{safe_name(target_col)}_{model_name}.png",
                    top_n=30,
                )

        except Exception as e:
            print(f"ERROR in model {model_name} for target {target_col}: {repr(e)}")
            all_model_summary_rows.append({
                "run_id": RUN_ID,
                "target_col": target_col,
                "model_name": model_name,
                "model_type": model_type,
                "base_name": base_name,
                "status": "failed",
                "error": repr(e),
            })

    # --------------------------------------------------------
    # Probability ensemble.
    # --------------------------------------------------------
    ensemble_name = "E1_valid_model_probability_ensemble"
    ensemble_cfg = MODEL_CONFIG[ensemble_name]

    if ensemble_cfg.get("enabled", True):
        print(f"\nTraining ensemble: {ensemble_name}")

        available_candidates = [
            m for m in ENSEMBLE_CANDIDATES
            if m in stored_probas[target_col]
        ]

        if len(available_candidates) >= 2:
            # Validation weights from macro-F1 and ordinal MAE.
            val_scores = []
            for m in available_candidates:
                val_proba = stored_probas[target_col][m]["validation"]
                val_pred = proba_to_pred(val_proba, labels=CLASS_LABELS)

                val_metrics = evaluate_predictions(
                    y_true=y_val.values,
                    y_pred=val_pred,
                    proba=val_proba,
                    labels=CLASS_LABELS,
                )

                # Positive score: prefer high macro-F1 and low ordinal MAE.
                score = max(val_metrics["macro_f1"], 0.0) / (1.0 + val_metrics["ordinal_mae"])
                val_scores.append(score)

            weights = np.asarray(val_scores, dtype=float)
            if weights.sum() <= 0:
                weights = np.ones(len(available_candidates)) / len(available_candidates)
            else:
                weights = weights / weights.sum()

            ensemble = ProbabilityEnsemble(available_candidates, weights)

            ensemble_weight_df = pd.DataFrame({
                "target_col": target_col,
                "ensemble_name": ensemble_name,
                "member_model": available_candidates,
                "weight": weights,
            })
            ensemble_weight_df.to_csv(
                METRIC_DIR / f"ensemble_weights_{safe_name(target_col)}_{ensemble_name}.csv",
                index=False,
            )

            start_time = time.time()

            split_proba_map = {}
            for split_name in ["train", "validation", "test"]:
                proba_dict = {
                    m: stored_probas[target_col][m][split_name]
                    for m in available_candidates
                }
                split_proba_map[split_name] = ensemble.combine(proba_dict)

            fit_seconds = time.time() - start_time

            ensemble_artifact = {
                "ensemble": ensemble,
                "available_candidates": available_candidates,
                "weights": weights,
                "target_col": target_col,
                "model_name": ensemble_name,
                "model_config": ensemble_cfg,
            }

            model_path = MODEL_DIR / f"model_{safe_name(target_col)}_{ensemble_name}.joblib"
            joblib.dump(ensemble_artifact, model_path)

            all_model_summary_rows.append({
                "run_id": RUN_ID,
                "target_col": target_col,
                "model_name": ensemble_name,
                "model_type": "probability_ensemble",
                "base_name": "ensemble",
                "status": "success",
                "fit_seconds": float(fit_seconds),
                "temperature": np.nan,
                "validation_log_loss_after_temperature": np.nan,
                "model_path": str(model_path),
            })

            model_prediction_frames = []
            model_probability_frames = []

            for split_name, (X_split, y_split, idx_split) in split_objects.items():
                proba = split_proba_map[split_name]
                y_pred = proba_to_pred(proba, labels=CLASS_LABELS)

                metric_values = evaluate_predictions(
                    y_true=y_split.values,
                    y_pred=y_pred,
                    proba=proba,
                    labels=CLASS_LABELS,
                )

                metric_row = {
                    "run_id": RUN_ID,
                    "run_timestamp_utc": RUN_TIMESTAMP,
                    "target_col": target_col,
                    "model_name": ensemble_name,
                    "model_type": "probability_ensemble",
                    "base_name": "ensemble",
                    "split": split_name,
                    "fit_seconds": float(fit_seconds),
                    "temperature": np.nan,
                    **metric_values,
                }

                all_metric_rows.append(metric_row)

                print(
                    f"{split_name:>10} | "
                    f"acc={metric_values['accuracy']:.4f} | "
                    f"bal_acc={metric_values['balanced_accuracy']:.4f} | "
                    f"macro_f1={metric_values['macro_f1']:.4f} | "
                    f"ord_mae={metric_values['ordinal_mae']:.4f} | "
                    f"qwk={metric_values['quadratic_weighted_kappa']:.4f} | "
                    f"ece={metric_values['ece_10bin']:.4f}"
                )

                cr_df = make_classification_report_df(
                    y_true=y_split.values,
                    y_pred=y_pred,
                    labels=CLASS_LABELS,
                )
                cr_df.insert(0, "split", split_name)
                cr_df.insert(0, "base_name", "ensemble")
                cr_df.insert(0, "model_type", "probability_ensemble")
                cr_df.insert(0, "model_name", ensemble_name)
                cr_df.insert(0, "target_col", target_col)
                cr_df.insert(0, "run_id", RUN_ID)
                all_class_report_rows.append(cr_df)

                cm = confusion_matrix(y_split.values, y_pred, labels=CLASS_LABELS)
                cm_df = pd.DataFrame(
                    cm,
                    index=[f"true_{c}" for c in CLASS_LABELS],
                    columns=[f"pred_{c}" for c in CLASS_LABELS],
                )
                cm_df.to_csv(
                    METRIC_DIR / f"confusion_matrix_{safe_name(target_col)}_{ensemble_name}_{split_name}.csv"
                )

                plot_confusion_matrix(
                    cm,
                    title=f"{target_col} | {ensemble_name} | {split_name}",
                    path=PLOT_DIR / f"confusion_matrix_{safe_name(target_col)}_{ensemble_name}_{split_name}.png",
                    labels=CLASS_LABELS,
                    normalize=False,
                )

                plot_confusion_matrix(
                    cm,
                    title=f"{target_col} | {ensemble_name} | {split_name} | normalized",
                    path=PLOT_DIR / f"confusion_matrix_normalized_{safe_name(target_col)}_{ensemble_name}_{split_name}.png",
                    labels=CLASS_LABELS,
                    normalize=True,
                )

                ece, cal_df = ece_multiclass(y_split.values, proba, n_bins=10)
                cal_df.insert(0, "split", split_name)
                cal_df.insert(0, "base_name", "ensemble")
                cal_df.insert(0, "model_type", "probability_ensemble")
                cal_df.insert(0, "model_name", ensemble_name)
                cal_df.insert(0, "target_col", target_col)
                cal_df.insert(0, "run_id", RUN_ID)
                all_calibration_rows.append(cal_df)

                cal_df.to_csv(
                    CALIBRATION_DIR / f"calibration_{safe_name(target_col)}_{ensemble_name}_{split_name}.csv",
                    index=False,
                )

                plot_reliability_curve(
                    cal_df,
                    title=f"{target_col} | {ensemble_name} | {split_name} | reliability",
                    path=PLOT_DIR / f"reliability_{safe_name(target_col)}_{ensemble_name}_{split_name}.png",
                )

                pred_df = pd.DataFrame(index=idx_split)
                pred_df.index.name = "date"
                pred_df["run_id"] = RUN_ID
                pred_df["target_col"] = target_col
                pred_df["model_name"] = ensemble_name
                pred_df["model_type"] = "probability_ensemble"
                pred_df["base_name"] = "ensemble"
                pred_df["split"] = split_name
                pred_df["y_true"] = y_split.values.astype(int)
                pred_df["y_pred"] = y_pred.astype(int)
                pred_df["abs_class_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])
                pred_df["squared_class_error"] = (pred_df["y_true"] - pred_df["y_pred"]) ** 2
                pred_df["expected_class"] = expected_class_from_proba(proba, labels=CLASS_LABELS)
                pred_df["expected_class_error"] = pred_df["y_true"] - pred_df["expected_class"]
                pred_df["max_probability"] = np.max(proba, axis=1)
                pred_df["entropy"] = predictive_entropy(proba)
                pred_df["normalized_entropy"] = normalized_entropy(proba)
                pred_df["probability_margin"] = probability_margin(proba)
                pred_df["ordinal_variance"] = ordinal_variance(proba, labels=CLASS_LABELS)
                pred_df["confidence_score"] = 1.0 - pred_df["normalized_entropy"]
                pred_df["is_correct"] = (pred_df["y_true"] == pred_df["y_pred"]).astype(int)
                pred_df["is_adjacent_or_correct"] = (pred_df["abs_class_error"] <= 1).astype(int)

                model_prediction_frames.append(pred_df)
                all_prediction_frames.append(pred_df)

                proba_df = pd.DataFrame(
                    proba,
                    index=idx_split,
                    columns=[f"proba_class_{c}" for c in CLASS_LABELS],
                )
                proba_df.index.name = "date"
                proba_df.insert(0, "split", split_name)
                proba_df.insert(0, "base_name", "ensemble")
                proba_df.insert(0, "model_type", "probability_ensemble")
                proba_df.insert(0, "model_name", ensemble_name)
                proba_df.insert(0, "target_col", target_col)
                proba_df.insert(0, "run_id", RUN_ID)

                model_probability_frames.append(proba_df)
                all_probability_frames.append(proba_df)

                if split_name == "test":
                    plot_uncertainty_error_relation(
                        pred_df,
                        title=f"{target_col} | {ensemble_name} | uncertainty vs error",
                        path=PLOT_DIR / f"uncertainty_error_{safe_name(target_col)}_{ensemble_name}_test.png",
                    )

                    plot_probability_heatmap(
                        proba_df,
                        title=f"{target_col} | {ensemble_name} | test probability heatmap",
                        path=PLOT_DIR / f"probability_heatmap_{safe_name(target_col)}_{ensemble_name}_test.png",
                    )

            pd.concat(model_prediction_frames, axis=0).to_parquet(
                PRED_DIR / f"predictions_{safe_name(target_col)}_{ensemble_name}.parquet"
            )
            pd.concat(model_prediction_frames, axis=0).to_csv(
                PRED_DIR / f"predictions_{safe_name(target_col)}_{ensemble_name}.csv"
            )

            pd.concat(model_probability_frames, axis=0).to_parquet(
                PROBA_DIR / f"probabilities_{safe_name(target_col)}_{ensemble_name}.parquet"
            )
            pd.concat(model_probability_frames, axis=0).to_csv(
                PROBA_DIR / f"probabilities_{safe_name(target_col)}_{ensemble_name}.csv"
            )

        else:
            print(f"Skipping ensemble for {target_col}: fewer than two candidate models available.")

# ============================================================
# 10. Aggregate saved outputs
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Saving aggregate metrics and artifacts")
print("=" * 80)

metrics_df = pd.DataFrame(all_metric_rows)
class_report_df = pd.concat(all_class_report_rows, ignore_index=True) if all_class_report_rows else pd.DataFrame()
predictions_df = pd.concat(all_prediction_frames, axis=0) if all_prediction_frames else pd.DataFrame()
probabilities_df = pd.concat(all_probability_frames, axis=0) if all_probability_frames else pd.DataFrame()
feature_importance_df = pd.concat(all_feature_importance_frames, ignore_index=True) if all_feature_importance_frames else pd.DataFrame()
calibration_df = pd.concat(all_calibration_rows, ignore_index=True) if all_calibration_rows else pd.DataFrame()
model_summary_df = pd.DataFrame(all_model_summary_rows)

metrics_df.to_csv(METRIC_DIR / "ordinal_uncertainty_metrics_all.csv", index=False)
metrics_df.to_parquet(METRIC_DIR / "ordinal_uncertainty_metrics_all.parquet", index=False)
metrics_df.to_csv(TABLE_DIR / f"table_18_ordinal_uncertainty_metrics_all_{RUN_ID}.csv", index=False)

if not class_report_df.empty:
    class_report_df.to_csv(METRIC_DIR / "ordinal_uncertainty_classification_reports_all.csv", index=False)
    class_report_df.to_parquet(METRIC_DIR / "ordinal_uncertainty_classification_reports_all.parquet", index=False)
    class_report_df.to_csv(TABLE_DIR / f"table_19_ordinal_uncertainty_classification_reports_all_{RUN_ID}.csv", index=False)

if not predictions_df.empty:
    predictions_df.to_parquet(PRED_DIR / "predictions_all_ordinal_uncertainty_models.parquet")
    predictions_df.to_csv(PRED_DIR / "predictions_all_ordinal_uncertainty_models.csv")

if not probabilities_df.empty:
    probabilities_df.to_parquet(PROBA_DIR / "probabilities_all_ordinal_uncertainty_models.parquet")
    probabilities_df.to_csv(PROBA_DIR / "probabilities_all_ordinal_uncertainty_models.csv")

if not feature_importance_df.empty:
    feature_importance_df.to_csv(IMPORTANCE_DIR / "feature_importance_all_ordinal_uncertainty_models.csv", index=False)
    feature_importance_df.to_parquet(IMPORTANCE_DIR / "feature_importance_all_ordinal_uncertainty_models.parquet", index=False)
    feature_importance_df.to_csv(TABLE_DIR / f"table_20_ordinal_feature_importance_all_{RUN_ID}.csv", index=False)

if not calibration_df.empty:
    calibration_df.to_csv(CALIBRATION_DIR / "calibration_all_ordinal_uncertainty_models.csv", index=False)
    calibration_df.to_parquet(CALIBRATION_DIR / "calibration_all_ordinal_uncertainty_models.parquet", index=False)
    calibration_df.to_csv(TABLE_DIR / f"table_21_calibration_all_ordinal_uncertainty_models_{RUN_ID}.csv", index=False)

model_summary_df.to_csv(METRIC_DIR / "ordinal_uncertainty_model_training_summary.csv", index=False)
model_summary_df.to_csv(TABLE_DIR / f"table_22_ordinal_model_training_summary_{RUN_ID}.csv", index=False)

# ============================================================
# 11. Leaderboards and plots
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating official leaderboards")
print("=" * 80)

leaderboards = []

for target_col in TARGET_COLS:
    for split_name in ["validation", "test"]:
        tmp = metrics_df[
            (metrics_df["target_col"] == target_col)
            & (metrics_df["split"] == split_name)
        ].copy()

        if tmp.empty:
            continue

        # Main composite rank:
        # - high macro-F1
        # - high balanced accuracy
        # - high QWK
        # - low ordinal MAE
        # - low ECE
        tmp["rank_macro_f1"] = tmp["macro_f1"].rank(ascending=False, method="min")
        tmp["rank_balanced_accuracy"] = tmp["balanced_accuracy"].rank(ascending=False, method="min")
        tmp["rank_qwk"] = tmp["quadratic_weighted_kappa"].rank(ascending=False, method="min")
        tmp["rank_ordinal_mae"] = tmp["ordinal_mae"].rank(ascending=True, method="min")
        tmp["rank_ece"] = tmp["ece_10bin"].rank(ascending=True, method="min")

        tmp["composite_rank"] = (
            tmp["rank_macro_f1"]
            + tmp["rank_balanced_accuracy"]
            + tmp["rank_qwk"]
            + tmp["rank_ordinal_mae"]
            + tmp["rank_ece"]
        ) / 5.0

        tmp = tmp.sort_values(
            ["composite_rank", "macro_f1", "balanced_accuracy", "quadratic_weighted_kappa", "ordinal_mae"],
            ascending=[True, False, False, False, True],
        )

        tmp.insert(0, "leaderboard_scope", f"{target_col}_{split_name}")
        leaderboards.append(tmp)

        print(f"\nLeaderboard: {target_col} | {split_name}")
        display_cols = [
            "model_name",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "quadratic_weighted_kappa",
            "ordinal_mae",
            "adjacent_accuracy_tol_1",
            "multiclass_log_loss",
            "multiclass_brier",
            "ece_10bin",
            "mean_normalized_entropy",
            "composite_rank",
        ]
        print(tmp[display_cols].to_string(index=False))

        tmp.to_csv(
            METRIC_DIR / f"leaderboard_{safe_name(target_col)}_{split_name}.csv",
            index=False,
        )

leaderboard_df = pd.concat(leaderboards, ignore_index=True) if leaderboards else pd.DataFrame()

if not leaderboard_df.empty:
    leaderboard_df.to_csv(METRIC_DIR / "leaderboards_all.csv", index=False)
    leaderboard_df.to_parquet(METRIC_DIR / "leaderboards_all.parquet", index=False)
    leaderboard_df.to_csv(TABLE_DIR / f"table_23_ordinal_uncertainty_leaderboards_{RUN_ID}.csv", index=False)

for target_col in TARGET_COLS:
    for metric_name, higher in [
        ("macro_f1", True),
        ("balanced_accuracy", True),
        ("quadratic_weighted_kappa", True),
        ("adjacent_accuracy_tol_1", True),
        ("ordinal_mae", False),
        ("ece_10bin", False),
        ("multiclass_brier", False),
        ("mean_normalized_entropy", False),
    ]:
        plot_metric_comparison(
            metrics_df,
            target_col=target_col,
            split_name="test",
            metric_name=metric_name,
            path=PLOT_DIR / f"model_comparison_{safe_name(target_col)}_test_{metric_name}.png",
            higher_is_better=higher,
        )

# ============================================================
# 12. Export best calibrated probabilities for Notebook 05
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Exporting allocation input probabilities")
print("=" * 80)

best_model_rows = []

for target_col in TARGET_COLS:
    tmp = leaderboard_df[
        (leaderboard_df["target_col"] == target_col)
        & (leaderboard_df["split"] == "validation")
    ].copy()

    if tmp.empty:
        print(f"No validation leaderboard found for {target_col}. Skipping allocation export.")
        continue

    best_row = tmp.sort_values("composite_rank", ascending=True).iloc[0]
    best_model = best_row["model_name"]

    print(f"Best validation model for {target_col}: {best_model}")

    # Export all splits for the best validation model.
    best_proba = probabilities_df[
        (probabilities_df["target_col"] == target_col)
        & (probabilities_df["model_name"] == best_model)
    ].copy()

    best_pred = predictions_df[
        (predictions_df["target_col"] == target_col)
        & (predictions_df["model_name"] == best_model)
    ].copy()

    if best_proba.empty or best_pred.empty:
        print(f"Missing probability or prediction rows for {target_col}, {best_model}.")
        continue

    best_proba.to_parquet(
        ALLOCATION_INPUT_DIR / f"allocation_regime_probabilities_{safe_name(target_col)}_{best_model}.parquet"
    )
    best_proba.to_csv(
        ALLOCATION_INPUT_DIR / f"allocation_regime_probabilities_{safe_name(target_col)}_{best_model}.csv"
    )

    best_pred.to_parquet(
        ALLOCATION_INPUT_DIR / f"allocation_regime_predictions_{safe_name(target_col)}_{best_model}.parquet"
    )
    best_pred.to_csv(
        ALLOCATION_INPUT_DIR / f"allocation_regime_predictions_{safe_name(target_col)}_{best_model}.csv"
    )

    # Test-only version for later backtest.
    best_proba_test = best_proba[best_proba["split"] == "test"].copy()
    best_pred_test = best_pred[best_pred["split"] == "test"].copy()

    best_proba_test.to_parquet(
        ALLOCATION_INPUT_DIR / f"allocation_regime_probabilities_TEST_{safe_name(target_col)}_{best_model}.parquet"
    )
    best_proba_test.to_csv(
        ALLOCATION_INPUT_DIR / f"allocation_regime_probabilities_TEST_{safe_name(target_col)}_{best_model}.csv"
    )

    best_pred_test.to_parquet(
        ALLOCATION_INPUT_DIR / f"allocation_regime_predictions_TEST_{safe_name(target_col)}_{best_model}.parquet"
    )
    best_pred_test.to_csv(
        ALLOCATION_INPUT_DIR / f"allocation_regime_predictions_TEST_{safe_name(target_col)}_{best_model}.csv"
    )

    best_model_rows.append({
        "target_col": target_col,
        "selected_by": "validation_composite_rank",
        "best_model": best_model,
        "validation_composite_rank": float(best_row["composite_rank"]),
        "validation_macro_f1": float(best_row["macro_f1"]),
        "validation_balanced_accuracy": float(best_row["balanced_accuracy"]),
        "validation_qwk": float(best_row["quadratic_weighted_kappa"]),
        "validation_ordinal_mae": float(best_row["ordinal_mae"]),
        "validation_ece_10bin": float(best_row["ece_10bin"]),
    })

best_model_selection_df = pd.DataFrame(best_model_rows)
best_model_selection_df.to_csv(
    ALLOCATION_INPUT_DIR / "selected_models_for_allocation.csv",
    index=False,
)
best_model_selection_df.to_csv(
    TABLE_DIR / f"table_24_selected_models_for_allocation_{RUN_ID}.csv",
    index=False,
)

print("\nSelected models for allocation:")
print(best_model_selection_df.to_string(index=False))

# ============================================================
# 13. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Validation report and file manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "04_AURORA_ordinal_imbalance_uncertainty_models.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "input_modeling_dataset": str(MODEL_DATA_PATH),
    "dataset_shape": df.shape,
    "date_range": {
        "start": str(df.index.min().date()),
        "end": str(df.index.max().date()),
    },
    "n_features": len(feature_cols),
    "target_cols": TARGET_COLS,
    "class_labels": CLASS_LABELS,
    "regime_label_definition": REGIME_LABEL_DEFINITION,
    "split_config": {
        "train_frac": TRAIN_FRAC,
        "validation_frac": VAL_FRAC,
        "test_frac": TEST_FRAC,
    },
    "split_report": split_report.to_dict(orient="records"),
    "model_config": MODEL_CONFIG,
    "ensemble_candidates": ENSEMBLE_CANDIDATES,
    "important_methodological_note": (
        "Previous-label persistence is excluded because overlapping future-return labels "
        "make y[t-1] unavailable at real-time prediction date t. Valid persistence-like "
        "features must be based on trailing realized returns only."
    ),
    "trailing_feature_cols": trailing_feature_cols,
    "n_metric_rows": int(len(metrics_df)),
    "n_prediction_rows": int(len(predictions_df)) if not predictions_df.empty else 0,
    "n_probability_rows": int(len(probabilities_df)) if not probabilities_df.empty else 0,
    "n_feature_importance_rows": int(len(feature_importance_df)) if not feature_importance_df.empty else 0,
    "n_calibration_rows": int(len(calibration_df)) if not calibration_df.empty else 0,
    "model_training_summary": model_summary_df.to_dict(orient="records"),
    "selected_models_for_allocation": best_model_selection_df.to_dict(orient="records"),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "metrics_dir": str(METRIC_DIR),
        "predictions_dir": str(PRED_DIR),
        "probabilities_dir": str(PROBA_DIR),
        "plots_dir": str(PLOT_DIR),
        "models_dir": str(MODEL_DIR),
        "calibration_dir": str(CALIBRATION_DIR),
        "allocation_input_dir": str(ALLOCATION_INPUT_DIR),
    },
}

save_json(RUN_ROOT / "AURORA_04_ordinal_uncertainty_validation_report.json", validation_report)
save_json(REPORT_DIR / f"AURORA_04_ordinal_uncertainty_validation_report_{RUN_ID}.json", validation_report)

manifest_df = make_file_manifest(RUN_ROOT)
manifest_df.to_csv(RUN_ROOT / "AURORA_04_ordinal_uncertainty_file_manifest_SHA256.csv", index=False)
manifest_df.to_csv(REPORT_DIR / f"AURORA_04_ordinal_uncertainty_file_manifest_SHA256_{RUN_ID}.csv", index=False)

# ============================================================
# 14. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF ORDINAL / IMBALANCE / UNCERTAINTY MODELS COMPLETE")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Run ID       :", RUN_ID)
print("Run root     :", RUN_ROOT)
print("Metrics      :", METRIC_DIR / "ordinal_uncertainty_metrics_all.csv")
print("Predictions  :", PRED_DIR)
print("Probabilities:", PROBA_DIR)
print("Calibration  :", CALIBRATION_DIR)
print("Allocation inputs:", ALLOCATION_INPUT_DIR)
print("Plots        :", PLOT_DIR)
print("Models       :", MODEL_DIR)
print("Manifest     :", RUN_ROOT / "AURORA_04_ordinal_uncertainty_file_manifest_SHA256.csv")
print("=" * 80)

print("\nFinal model training summary:")
print(model_summary_df.to_string(index=False))

print("\nSaved key paper tables:")
print(TABLE_DIR / f"table_16_ordinal_split_report_{RUN_ID}.csv")
print(TABLE_DIR / f"table_18_ordinal_uncertainty_metrics_all_{RUN_ID}.csv")
print(TABLE_DIR / f"table_19_ordinal_uncertainty_classification_reports_all_{RUN_ID}.csv")
print(TABLE_DIR / f"table_21_calibration_all_ordinal_uncertainty_models_{RUN_ID}.csv")
print(TABLE_DIR / f"table_23_ordinal_uncertainty_leaderboards_{RUN_ID}.csv")
print(TABLE_DIR / f"table_24_selected_models_for_allocation_{RUN_ID}.csv")

print("\nNext notebook:")
print("05_AURORA_uncertainty_aware_etf_allocation.ipynb")

Mounted at /content/drive
AURORA-TWETF Ordinal, Imbalance-Aware, Uncertainty Models
Timestamp UTC: 2026-06-23T15:19:20Z
Run ID       : 20260623_151920
Project root : /content/drive/MyDrive/AURORA_TWETF
Input data   : /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
Run root     : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/ordinal_imbalance_uncertainty/run_20260623_151920

Step 1: Loading modeling dataset
Original dataset shape: (1262, 417)
Original date range   : 2021-01-06 to 2026-03-25
No TAIEX close column found. Skipping trailing realized regime features.
No trailing feature warm-up rows dropped.
Final loaded dataset shape: (1262, 417)
Final date range          : 2021-01-06 to 2026-03-25
Feature columns           : 415
Target columns            : ['TAIEX_regime_fixed_20d', 'TAIEX_regime_fixed_60d']

Step 2: Chronological train / validation / test split
     split   n start_date   end_date
     train 883 2021-01-06 2024-08-27
